# Dynamic Portfolio Management

*A Stochastic Programming Model for Dynamic Portfolio Management with Derivatives*
___
___

### Notes
- For deep hedging proposed method, use full surface as factor input, only trade few options (for interpretability and realism): few different strikes and maturities
- Buy deep ITM call options for full exposure to the asset but with current liquidity

___
### Patterns

1. Build the tree
    - Each node knows 4 stochastic values (equity, bond, money market, VIX) + a risk free rate (deterministic) used separately.
    - The 4 stochastic assets get bootstrapped to build the tree: for each node, a children is created by sampling a historical return vectors of these assets: to keep there historical correlation. 
    - Each node must have at least 4 children to solve non-arbitrage conditions. The paper uses monthly nodes for 6 months, and the first node has 10 childen, that is, total number of nodes is $\small 1 + 10 + 10 \ (4) + 10 \ (4^2) + 10 \ (4^3) + ...$
    - Then, we have a full arbitrage-free tree of asset prices under the physical measure.


2. Price derivatives on that tree
    - Once we have the full arbitrage-free tree of asset prices with physical measure $p(n)$, fit the minimal-entropy risk-neutral measure $q(n)$ node-by-node (same tree, different probabilities), then get the option prices at every node by backward induction under risk-neutral $q$

3. Solve the portfolio LP
    - By this point, every price on every node — assets and options alike — is a fixed number, computed and done. The only remaining unknowns are the trading decisions ($x_{in}$, $c^h_{1n}(j,k)$, etc.). Solve

        $$\max_{x,c,p} \,\,\,\, (1-\lambda) \ \mathbb E[W_{N_T}] - \lambda \ \text{CVaR}_{\alpha}(W_{N_T})$$

        using the physical probabilities $p(n)$, subject to the wealth / cash / inventory-balance /position-bound constraints, plus whichever strategy-shape constraints you're imposing if any (protective put / covered call / straddle, ...).

4. Rolling / Receding-horizon re-optimization
    - At $t=0$, run steps 1,2,3, get the decision and execute it.
    - Let one period pass, now at $t=1$ observe the actual realized market return
    - At this new date, rebuild a fresh tree from stage 1 (using updated historical data up to now), re-price derivative (stage 2), re-solve LP (stage 3), then get a new decision and execute it
    - Repeat

5. Evaluation (Out-of-sample check)
    - After solving, find whichever path is closest (Euclidean distance) to what market actually did, and read off that path's node-level decisions to mark-to-market the strategy at realized prices. Roll forward to test on longer period, using period final wealth as the next period initial wealth.
    - This checks whether the built tree was good or not (even if the policy is optimal on that tree, the tree might be a misrepresentation of the market)

___
### Model Overview


**Advantages**
- Reshape historical return distribution
- Empirically strong results
- Regime-adaptive sizing, not a fixed rule
    - Optimizer decides when, how much and which strikes/maturities
- Tunable risk appetite
    - $\lambda$ gives a single and interpretable choice between growth and risk
- Fully interpretable
- Exact optimum from stated objective
    - LP solver finds optimal policy
- Hard constraints are exactly enforced 

**Disadvantages**
- Curse of dimensionality
- Policy is optimal for that one finite tree
    - The tree might be a misrepresentation of the market
- Historical bootstrap dependence
- No transferable policy function
    - Every rebalancing require new solving, cannot learn a policy and evaluate instantly like a trained neural policy
- Option pricing under incompleteness is a modeling choice
    - MEM/entropy measure is one defensible way to pick among the infinitely many risk-neutral measures consistent with no-arbitrage in an incomplete market.